Multiple tasks/workflows at the same time. Fan-In and Fan-Out pattern. Single node splits the flow into 2 jor more independent branches that execute concurrently. Later these branches merge back into a single point called as join node before the graph completes.<br>
When graph splits, multiple nodes are writing to the state simultaneouly without overwriting each others data.

Automated AI content moderation and brand safety pipeline:<br>
Raw text/script/comment/blog sent to 3 agents that simultaneously evaluate on different perspective and score it from 1-100.
Toxicity monitor, Cultural sensitivity and Copyright.

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()
from typing import Annotated
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

In [19]:
model = ChatGroq(
    model = 'openai/gpt-oss-120b',
    temperature = 0.1,                  # coz i dont need scoring creativity
    max_completion_tokens = 200
)

/home/rageshwer-singh/dev/projects/LangGraph/.venv/lib/python3.12/site-packages/pydantic/main.py:280: UserWarning: WARNING! max_completion_tokens is not default parameter.
                    max_completion_tokens was transferred to model_kwargs.
                    Please confirm that max_completion_tokens is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


See, we want one safety score of a text. We can make pydantic object with 3 different scores and make some formula at the end to rate the safety. But here we want that only one score is maintained. So, when 3 branches work continuously, this safety score will be overwritten, so we will use the concept of reducers. First make the dict of safety scores.

In [20]:
def merge_score_dicts(existing : dict, updated : dict) -> dict:
    if existing is None:
        return updated
    return {**existing,**updated}
# State :
class State(BaseModel):
    raw_text : str
    safety_score : Annotated[dict[str , int],merge_score_dicts]


In [23]:
# Nodes :
def toxicity(state : State) -> dict:
    '''Branch 1 : Checks the text for toxicity level.'''
    prompt = f'''
        Analyze the text for profanity, aggression, hate speech or toxicity.
        Provide only an integer score from 0-100 where 0 is clean and 100 is 
        highly toxic. Text : {state.raw_text}
        '''
    
    response = model.invoke(prompt)
    return {
        'safety_score':{'toxicity_score':int(response.content)}
    }

def cultural(state : State) -> dict:
    '''Branch 2 : Checks the text for cultural remarks.'''
    prompt = f'''
        Analyze the text for harmful, racist or hateful cultural remarks.
        Provide only an integer score from 0-100 where 0 is clean and 100 is 
        highly targeted. Text : {state.raw_text}
        '''
    
    response = model.invoke(prompt)
    return {
        'safety_score':{'cultural_score':int(response.content)}
    }

def copyright(state : State) -> dict:
    '''Branch 3 : Checks the text for copyright infringement.'''
    prompt =f'''
        Analyze the text for any copyright issues. Assess that text
        is original or plagiarized. Provide only an integer score from 0-100.
        A score of 0 means the text is original and 100 means totally copied.
        Text : {state.raw_text}
        '''
    
    response = model.invoke(prompt)
    return {
        'safety_score':{'copyright_score':int(response.content)}
    }

In [24]:
# Edges
builder = StateGraph(State)
builder.add_node('toxic',toxicity)
builder.add_node('cultural',cultural)
builder.add_node('copyright',copyright)

builder.add_edge(START,'toxic')
builder.add_edge(START,'cultural')      # Fan-Out
builder.add_edge(START,'copyright')

builder.add_edge('toxic',END)
builder.add_edge('cultural',END)
builder.add_edge('copyright',END)

graph = builder.compile()

response = graph.invoke({
    'raw_text':"I hate black people",
    'safety_score':{}
})

print(response.get('safety_score'))

{'copyright_score': 0, 'cultural_score': 100, 'toxicity_score': 95}
